# MLflow 5 minute Tracking Quickstart

This notebook demonstrates using a local MLflow Tracking Server to:

- log,  
- register, and  
- load a model as a generic Python Function (pyfunc) to perform inference on a Pandas DataFrame.

Throughout this notebook, we'll be using the MLflow fluent API to perform all interactions with the MLflow Tracking Server.

This is a modification of the official MLflow quickstart: **[MLflow Tracking Quickstart](https://mlflow.org/docs/latest/ml/getting-started/quickstart/)**.   
Where this notebook corrects, supplements, or deliberately diverges from upstream, the relationship is named with an inline bold callout — one of **Bug**, **Stale**, **Missing from**, or **Diverges from upstream tutorial:** — or, in code cells, with a `NOTE (differs from upstream)` comment.  
Sections with no upstream counterpart use a plain heading and no callout.

Two structural differences worth knowing up front:

- **Training happens outside `start_run()`:**  
Upstream calls `LogisticRegression(...).fit(...)` *inside* the `with mlflow.start_run():` block.   
This notebook trains first (in the "Load training data" cell below) and opens a run only to log the already-trained model, so that a training error never leaves a half-populated run on the server. The "Log the model" section explains why.  
- **Manual logging only — no autolog:**  
Upstream's Step 3 demonstrates `mlflow.autolog()`, which automatically captures hyperparameters, metrics, and the model when you call `fit`.   
This notebook uses explicit `log_params` / `log_metric` / `log_model` calls so you can see exactly what gets recorded.  
Switch to autolog once you understand the manual API — see the **The one-line alternative: `mlflow.autolog()`** section below.

### Set the MLflow Tracking URI 

Depending on where you are running this notebook, your configuration may vary for how you initialize the interface with the MLflow Tracking Server. 

For this example, we're using a locally running tracking server, but other options are available.  
(The easiest is to use the free managed service within the [Databricks Free Trial](https://mlflow.org/docs/latest/getting-started/databricks-trial.html)). 

Please see [the guide to running notebooks here](https://www.mlflow.org/docs/latest/getting-started/running-notebooks/) for more information on setting the tracking server uri and configuring access to either managed or self-managed MLflow tracking servers.

## Start the tracking server first

**Missing from upstream tutorial:**  
The official quickstart calls `mlflow.set_tracking_uri("http://127.0.0.1:5000")` without ever explaining that **something has to be listening on that URI**.  
But, if the server is not running, every logging cell below fails: `mlflow.start_run()` raises a connection error wrapped in an `MlflowException`, and the "View run" / "View experiment" links printed at the end of each run return *connection refused*.

### Why this needs to come first

A *tracking server* is the process that owns:  
- the experiment metadata database,  
- the artifact store, and  
- the model registry.   

The notebook only ever talks to it over HTTP — there is no in-process fallback.   
So before any other cell in this notebook works, you need to start one:

```bash
cd src/
mlflow ui --host 127.0.0.1 --port 5000   # use 5001, 5002, … if 5000 is taken
```

Run it **from `src/`, in a separate terminal**, and **leave it running** for the rest of the notebook.

A few things worth knowing about that one command:

- **`mlflow ui` is the single-user, local-defaults variant of `mlflow server`** — same code path, no auth, no LAN exposure.  
It is the right choice for tutorials and local experimentation, not for shared deployments.  
- **Why `src/` and not the repo root.** The server creates `mlflow.db` and an `mlartifacts/` directory in its working directory.   
Putting them under `src/` keeps per-developer runtime state inside the source tree (alongside the notebooks under `src/`) rather than scattering it at the project root next to `pyproject.toml`, `README.md`, etc.  
(If you previously started the server from the repo root, you may also have a stray `mlflow.db` there — safe to `rm`, it's per-developer state.)  
- **Port 5000 may be taken.** Pass `--port 5001` (or any free port) and update `PORT` in the `mlflow.set_tracking_uri(...)` cell below to match.

### Confirm it's up

Open <http://127.0.0.1:5000/> in your browser.   
You should see an empty MLflow UI with a `Default` experiment.  
*Connection refused* means the server isn't running; a different port in the terminal startup log means use that port instead.

### Verify the on-disk layout with `tree`

After the first logging cell further down runs successfully, you can sanity-check what the server wrote.   
From the **`src/`** directory (where you started the server):

```bash
tree mlflow.db mlartifacts/
```

You'll see `mlflow.db` (a single file) and an `mlartifacts/<experiment_id>/models/m-<uuid>/artifacts/` subtree. A worked example of what to expect appears in the "Directory structure" cell after the first run cell. If `tree` isn't installed, `find mlartifacts -maxdepth 4` is a flat-format substitute.

## MLflow Default Database and Registry store URI

When you launch `mlflow ui` (or `mlflow server`) without flags, MLflow has to pick defaults for **three** independent stores.   
Understanding what each one holds — and which defaults changed in MLflow 3 — keeps the rest of this tutorial from feeling like magic.

### The three stores

| Store | Holds | Default URI (MLflow 3+) |
|---|---|---|
| **Backend (tracking) store** | Experiments, runs, parameters, metrics, tags — all *metadata* | `sqlite:///mlflow.db` |
| **Artifact store** | Model files, plots, datasets, anything binary you log | `mlflow-artifacts:` (proxied) → `./mlartifacts/` on disk |
| **Model Registry store** | Registered models, versions, aliases, stages | Same URI as the backend store |

You only typically interact with these defaults by *not* setting them.   
Each can be overridden with a CLI flag (`--backend-store-uri`, `--default-artifact-root`, `--registry-store-uri`) or with an environment variable.

### MLflow Database (Backend Store)

**Default since MLflow 3+: `sqlite:///mlflow.db`**

The backend store is where MLflow records the *metadata* about your experiments and runs:  
- the run id,  
- start/end time,  
- parameters you logged,  
- metrics,  
- tags,  
- status,  
- the user, and so on.

It does **not** hold model files — those go to the artifact store.

**What changed in MLflow 3:**  
prior versions defaulted to `file:./mlruns`, a flat directory of YAML/text files (one folder per experiment, one per run).   

It worked, but it had two problems:

1. **No concurrent writes.** Two processes writing runs at the same time could corrupt the directory.  
2. **No Model Registry support.** The registry needs SQL-style transactions (atomic version creation, foreign keys between models and runs).      
The file backend can't provide that, so `mlflow.register_model(...)` and `registered_model_name=...` simply failed with the file backend.

From MLflow 3 onward, `mlflow ui` creates a SQLite database file called `mlflow.db` in the working directory where you started the server. 

SQLite gives you transactions and lets the registry "just work" out of the box.  
This is why the `registered_model_name="tracking-quickstart"` argument later in this notebook succeeds without any extra configuration.

**Where the file lives:** in the cwd of the `mlflow ui` process. If you start the server from the repo root, you'll see `mlflow.db` appear there.   

Add it to `.gitignore` — it's per-developer state, not source code.

**Inspecting it (optional):**
```bash
sqlite3 mlflow.db ".tables"
sqlite3 mlflow.db "SELECT experiment_id, name FROM experiments;"
```

**Overriding it:** pass `--backend-store-uri` to use Postgres / MySQL / a different SQLite path:
```bash
mlflow ui --backend-store-uri postgresql://user:pw@host:5432/mlflow
mlflow ui --backend-store-uri sqlite:///path/to/custom.db
```

### MLflow Registry Store

**Default: same URI as the backend store** (so also `sqlite:///mlflow.db` by default in MLflow 3+).

The Model Registry is a separate logical service that:  
- tracks *registered models* — named, versioned references to logged models, plus their aliases (e.g. `champion`, `challenger`) and lifecycle stage.    

A "run" logs a model artifact; "registering" it gives that artifact a stable name and version number you can promote, query, and load by name from production code.

The registry needs a database backend for the same transactional reasons as above.   
MLflow's design choice is: rather than configure two URIs, default the registry to **share** the backend's URI.   
Both end up as tables inside the same SQLite file (`registered_models`, `model_versions`, etc., alongside `experiments`, `runs`, `params`, `metrics`).

You can split them if you ever need to — e.g. a shared team-wide registry but a per-user tracking backend — by passing `--registry-store-uri` explicitly:
```bash
mlflow server \
  --backend-store-uri sqlite:///local-tracking.db \
  --registry-store-uri postgresql://team@registry-host/mlflow_registry
```
For local learning, leaving the defaults alone is the right call.

### Reading the server's startup log

The lines `mlflow ui` prints on startup map directly onto the table above:

```text
Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only).
To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
```

- **Line 1** — backend store fell through to the MLflow 3+ default, SQLite.  
- **Line 2** — registry store was not separately configured, so it shares the backend URI.  
- **Lines 3–4** — unrelated to storage: MLflow 3+ binds the UI to `127.0.0.1` only and rejects requests with non-localhost `Host:` headers as a SSRF/CSRF hardening default. If you `--host 0.0.0.0` to expose the UI on a LAN, you also need to opt into trusted hostnames and origins.

### What ends up in git (or doesn't)

`mlflow.db`, `mlartifacts/`, and `mlruns/` are all per-developer runtime state, not source code. Add them to `.gitignore`:

```gitignore
mlflow.db
mlartifacts/
mlruns/
```

The "Directory structure" cells later in the notebook show what the on-disk layout actually looks like after running a few cells.

### Sources

- [Use SQLite as default unless existing mlruns data is detected — mlflow/mlflow PR #18497](https://github.com/mlflow/mlflow/pull/18497)  
- [Backend Stores — MLflow docs](https://mlflow.org/docs/latest/ml/tracking/backend-stores/)

In [1]:
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

In [2]:
# NOTE: review the links mentioned above for guidance on connecting to a managed tracking server, such as the Databricks Managed MLflow
HOST = "127.0.0.1"
PORT = 5001
mlflow.set_tracking_uri(uri=f"http://{HOST}:{PORT}")

> **GenAI readers — you can skim from here.**  
Everything above — the tracking server, experiments, runs, and the three stores — applies to *any* MLflow project, GenAI included.  
The rest of this notebook logs a **scikit-learn** model as the worked example: `log_model`, then loading it back as a `pyfunc`. That pyfunc round-trip is shared by both tracks (a GenAI app is also logged and loaded as a pyfunc). The scikit-learn–specific deep dives — safe `skops` serialization and `mlflow.sklearn.autolog()` — live in `ml/a_model_logging`. The GenAI track captures models and traces differently; see `gen_ai/` (and `roadmap/`).

## Load training data and train a simple model

For our quickstart, we're going to be using the familiar iris dataset that is included in scikit-learn.   

Following the split of the data, we're going to train a simple logistic regression classifier on the training data and calculate some error metrics on our holdout test data. 

Note that the only MLflow-related activities in this portion is the fact that we're using a `param` dictionary to supply our model's hyperparameters.   
This is to make logging these settings easier when we're ready to log our model and its associated metadata.

Nothing is logged to MLflow in this loading of data and in this training.

In [3]:
# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Define the model hyperparameters
# NOTE (differs from upstream tutorial): the original `params` dict includes
# `"multi_class": "auto"`. That keyword was deprecated in scikit-learn 1.5 and
# removed in 1.7, so it raises `TypeError` on scikit-learn >= 1.7. Modern
# LogisticRegression automatically uses multinomial for multiclass — drop the keyword.
params = {"solver": "lbfgs", "max_iter": 1000, "random_state": 8888}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

# Calculate accuracy as a target loss metric
accuracy = accuracy_score(y_test, y_pred)

## Define an MLflow Experiment

In order to group any distinct runs of a particular project or idea together, we can define an Experiment that will group each iteration (runs) together.  

Define a unique name relevant to what we're working on, to help with organization and reduce the amount of work (searching) to find our runs later on. 

In [5]:
experiment_name = "MLflow Quickstart with Logistic Regression"

mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1779434899989, experiment_id='1', last_update_time=1779434899989, lifecycle_stage='active', name='MLflow Quickstart with Logistic Regression', tags={}, trace_location=None, workspace='default'>

In [6]:
mlflow.get_experiment_by_name(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1779434899989, experiment_id='1', last_update_time=1779434899989, lifecycle_stage='active', name='MLflow Quickstart with Logistic Regression', tags={}, trace_location=None, workspace='default'>

You may verify in the UI that a new experiment has been created with that name.

## Log the model, hyperparameters, and loss metrics to MLflow.

To record:  
- our model,  
- the hyperparameters that were used when fitting it, and  
- the metrics from validating it on the holdout data,

we initiate a **run context**, as shown below.

Within the scope of that context, any fluent API call (e.g. `mlflow.log_params()` or `mlflow.sklearn.log_model()`) is associated and logged together to the same run.

> **Footnote — what is a "fluent API"?**  
>  
> **In general.**  
> "Fluent API" is a design-style term — coined by [Martin Fowler](https://martinfowler.com/bliki/FluentInterface.html) — for an interface that reads close to natural language because you don't have to keep re-passing the same arguments.  
> The familiar shape is **method chaining**: `query.filter(...).order_by(...).limit(10)`.  
> The broader pattern, and the one MLflow uses, is **top-level calls that act on an ambient piece of state the library manages for you**.  
>  
> **In MLflow.**  
> That ambient state is the **active run**.  
> `mlflow.start_run()` opens a run and stashes it on a thread-local; until the `with` block exits, plain top-level calls like `mlflow.log_param(...)`, `mlflow.log_metric(...)`, or `mlflow.sklearn.log_model(...)` quietly attach to it — you never spell out the `run_id` yourself.  
> The same idea applies to the active *experiment* (`mlflow.set_experiment(...)`) and the active *tracking URI* (`mlflow.set_tracking_uri(...)`).  
>  
> **The non-fluent alternative — and *no*, it is not the same thing.**  
> MLflow also ships an **explicit** API called `mlflow.client.MlflowClient` (often just "the client API") for the same operations, where every call takes the relevant ids as arguments.  
> The fluent API and the client API are **opposites**, not synonyms — easy to mix up because both end up doing the same logging under the hood:  
>  
> | | Fluent API | Client API |  
> |---|---|---|  
> | Where you import from | `import mlflow` | `from mlflow.client import MlflowClient` |  
> | How you call it | Top-level functions: `mlflow.log_param("alpha", 0.1)` | Methods on an instance: `client.log_param(run_id, "alpha", 0.1)` |  
> | Where the `run_id` comes from | Implicit — the active run set by `mlflow.start_run()` | Explicit — you pass it on every call |  
> | Best for | Training scripts and notebooks, where there's a clear "current" run | Tooling, batch backfills, cross-process pipelines |  
>  
> The two are equivalent in capability.  
> This tutorial uses the **fluent** API throughout (shorter, less error-prone for a linear notebook).  
> Production pipelines and MLflow's own UI internals tend to use the **client** API.

**Where the run's data will land.** Two paths get written every time we log a run:

- **The metadata** (parameters, metrics, tags, run id, experiment id, status) goes into the **server's backend store**.  
  By default that's `sqlite:///mlflow.db` in the **server's working directory** — whichever directory you launched `mlflow ui` / `mlflow server` from.  
  Not the notebook's cwd. Not the repo root, unless you happened to start the server there.  
- **The model artifact** (`MLmodel`, `model.skops`, `conda.yaml`, …) goes under `mlartifacts/<experiment_id>/models/m-<uuid>/artifacts/`, again relative to the server's cwd.

If you're unsure where the server's cwd is, scroll back through its terminal for this startup line:

```text
Backend store URI not provided. Using sqlite:///mlflow.db
```

That path is relative to *that* shell's working directory at launch time. Similarly:

```bash
ls mlflow.db mlartifacts/
```

run from the same directory will confirm both exist after the cell below executes.

### Which experiment will a `run` attach to?

The `mlflow.start_run()` call below takes **no** experiment argument — yet the resulting run lands under `"MLflow Quickstart with Logistic Regression"`, the experiment we set a couple of cells ago.

That's the fluent API at work again:  
`mlflow.set_experiment(name)` updates the **active experiment** — a process-global piece of state on the client — and every subsequent `start_run()` reads from it until something else changes it.  
(Concretely: the active experiment is stored on a thread-local; if you never `set_experiment(...)`, runs go into `Default` / experiment id `0`.)

#### Using a different experiment — including one created weeks ago

Just call `set_experiment` again before `start_run`:

```python
mlflow.set_experiment("legacy-research-q2-2024")  # created weeks ago

with mlflow.start_run():
    mlflow.log_metric("accuracy", 0.91)
```

A few details that come up in practice:

- **If the experiment already exists:**  
  `set_experiment` silently re-attaches to it — no warning, no duplicate, no new id.  
  Lookup is by name; the integer `experiment_id` keeps whatever value it was assigned when the experiment was first created.

- **If it doesn't exist:**  
  `set_experiment` *creates* it on the fly — the same `INFO mlflow.tracking.fluent: Experiment ... does not exist. Creating ...` line we saw earlier.  
  Convenient in a notebook; **dangerous in a production script** — a typo in the experiment name silently forks your runs into a brand-new parallel experiment.  
  To force "must already exist":
  ```python
  exp = mlflow.get_experiment_by_name("legacy-research-q2-2024")
  if exp is None:
      raise RuntimeError("Experiment 'legacy-research-q2-2024' not found")
  mlflow.set_experiment(experiment_id=exp.experiment_id)
  ```

- **If you only have the id** (e.g. from a UI URL like `/experiments/7`):  
  `set_experiment` accepts it directly: `mlflow.set_experiment(experiment_id="7")`.

- **One-off run without flipping the global state:**  
  pass `experiment_id` straight to `start_run`. The active experiment stays whatever it was:
  ```python
  with mlflow.start_run(experiment_id="7"):
      mlflow.log_metric("accuracy", 0.91)
  ```
  Note that `start_run` takes `experiment_id` but **not** `experiment_name`.  
  If you only have a name and don't want to flip the active experiment, look up the id first: `get_experiment_by_name(name).experiment_id`.

In [7]:
# NOTE (differs from upstream tutorial): training happened in the cell above,
# outside any run context — keeping fit() out of `start_run()` avoids leaving
# a half-populated run on the server if training fails. Inside the with-block
# we only LOG the artifacts that already exist.
#
# NOTE (differs from upstream tutorial): serialization_format="skops" avoids
# pickle/cloudpickle, which execute arbitrary code on load. See `ml/a_model_logging`
# for the full explanation.

with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("accuracy", accuracy)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    # Infer the model signature (input/output schema) from a sample
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the trained model — also registers it in the Model Registry
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        name="iris_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="tracking-quickstart",
        serialization_format="skops",
    )

2026/05/22 10:40:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2026/05/22 10:40:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 6
Created version '6' of model 'tracking-quickstart'.


🏃 View run vaunted-newt-644 at: http://127.0.0.1:5001/#/experiments/1/runs/134f9cff3d0d4f10bc237fda6eb22bcb
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


### After the run

**View it in the UI.**  
Open <http://127.0.0.1:5000/> (or whichever port you started `mlflow ui` on).  
Under the `MLflow Quickstart with Logistic Regression` experiment you should see:

- one run named after a randomly-generated phrase (`blushing-rook-969` etc.),  
- with the `accuracy` metric, the `solver` / `max_iter` / `random_state` params, and the `Training Info` tag,  
- plus the logged model.

The "Models" tab should also list a registered model called `tracking-quickstart` at version 1.

**Inspect it on disk.**  
Run `tree mlartifacts/ mlflow.db` from `src/` to reproduce:

```text
src/
├── mlartifacts/
│   └── 1/                                  ← experiment_id (integer, not name)
│       └── models/
│           └── m-260866c0927a4086abb797184f736ea3/
│               └── artifacts/
│                   ├── MLmodel
│                   ├── model.skops          ← serialized model (skops, not pickle)
│                   ├── conda.yaml
│                   ├── input_example.json
│                   ├── python_env.yaml
│                   ├── requirements.txt
│                   └── serving_input_example.json
└── mlflow.db                                ← SQLite backend store
```

This is the only tree dump in the notebook — everything else MLflow writes builds on this shape. Two things to notice:

- **The directory next to `models/` is the integer `experiment_id`** (`1` here, because `Default` was `0`).  
  MLflow uses the id, not the name, so renaming an experiment doesn't move thousands of files.  
  The "Naming the artifact directory" section further down shows how to override this with the `artifact_location` parameter.

- **`m-<32 hex chars>` is a LoggedModel UUID.**  
  Every `mlflow.sklearn.log_model(...)` call creates a new one — re-running the cell above produces a sibling `m-<new-uuid>/` directory next to this one, never overwriting. (Your own UUID will be different; mlflow generates a fresh one per call.)  
  The `tracking-quickstart` registered-model row in the registry advances to version 2, 3, … on each re-run — see the "Why version 2?" cell below.

**Why training and logging are split across two cells.**  
It's tempting to put `LogisticRegression(...).fit(...)` inside the `with start_run():` block — that's what the upstream quickstart does.  
It works, but if training raises (bad hyperparameters, OOM, anything) you end up with an empty or half-populated run on the server that you then have to clean up manually.  
Doing `fit` first and only logging *already-materialized* artifacts inside the run context avoids that whole class of mess.

## Load our saved model as a Python Function

We can load our model back as a native scikit-learn format with `mlflow.sklearn.load_model()`.   

But,  we are loading the model as a generic Python Function, which is how this model would be loaded for online model serving.    

We can still use the `pyfunc` representation for batch use cases, though, as is shown below.

In [8]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

## Use our model to predict the iris class type on a Pandas DataFrame

In [9]:
predictions = loaded_model.predict(X_test)

iris_feature_names = datasets.load_iris().feature_names

# Convert X_test validation feature data to a Pandas DataFrame
result = pd.DataFrame(X_test, columns=iris_feature_names)

# Add the actual classes to the DataFrame
result["actual_class"] = y_test

# Add the model predictions to the DataFrame
result["predicted_class"] = predictions

result[:4]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
0,6.1,2.8,4.7,1.2,1,1
1,5.7,3.8,1.7,0.3,0,0
2,7.7,2.6,6.9,2.3,2,2
3,6.0,2.9,4.5,1.5,1,1


### Safe serialization (`skops`)

The `log_model` call above passes `serialization_format="skops"`. By default MLflow serializes scikit-learn models with `cloudpickle`, which can execute arbitrary code on load (and emits a security warning); `skops` is the safe alternative.

This is a scikit-learn–specific concern, so the full explanation — why pickle is unsafe, how the `skops` format differs, the custom-class caveat, and when *not* to use it — lives in **`ml/a_model_logging`**. For this quickstart, just know the keyword silences the warning and makes the registered model safe for others to load.

## Naming the artifact directory with `artifact_location`

Earlier — see the "After the run" cell — you saw that the on-disk layout under `mlartifacts/` uses an **integer** as the parent directory name (`mlartifacts/1/…`), not the experiment's human-readable name.  
That integer is the `experiment_id`.

This addendum explains:

1. **why** the parent directory is the integer `1` and not the experiment's name,  
2. **which MLflow parameter** controls it — `artifact_location` on `create_experiment`, and  
3. **how** to opt into a human-readable path on a *new* experiment without disturbing anything the main tutorial already wrote.

The main tutorial above is left intact on purpose — the integer-id layout is what MLflow gives you out of the box, and seeing it once is part of the lesson.

### Why the integer `1`?

When you called `mlflow.set_experiment("MLflow Quickstart with Logistic Regression")` earlier and that experiment didn't yet exist, MLflow created it and assigned the next available **integer id**.  
The built-in `Default` experiment is always `0`; yours was the second to exist on this tracking server, so it got `1`.  
The output of the `set_experiment` cell makes this explicit:

```text
<Experiment: artifact_location='mlflow-artifacts:/1', experiment_id='1',
             name='MLflow Quickstart with Logistic Regression', ...>
```

The `artifact_location` is the **per-experiment root** under which all runs and their models live.

MLflow uses `experiment_id` (not the name) in this path on purpose:

- Experiments can be renamed; the id is stable.  
- Putting the name into the on-disk path would mean a rename has to move thousands of files.  
- Using the id keeps rename a one-row `UPDATE` in `mlflow.db`.

### Why repeated runs share `1/`

`mlflow.set_experiment("MLflow Quickstart with Logistic Regression")` was called once.  
Every subsequent `mlflow.start_run()` (re-executing the log-model cell) attached a new run to the *same* experiment, so:

- Same experiment → same `experiment_id` → same artifact root → same `1/` folder.

### Why each `m-<hash>` is a separate directory

In MLflow 3, every call to `mlflow.<flavor>.log_model(...)` creates a **`LoggedModel` entity** with its own UUID, separate from the run that produced it.  
That UUID is the `m-<hex>` in the path.  
Two runs that each log a model produce two `LoggedModels` with two different UUIDs and therefore two sibling directories under `models/`.  
The files inside each (`MLmodel`, `conda.yaml`, `requirements.txt`, …) are regenerated per `LoggedModel`.

### The MLflow parameter responsible: `artifact_location`

The parameter that controls this on-disk path is **`artifact_location`** on `mlflow.create_experiment(name, artifact_location=..., tags=...)`.

- **If you don't pass it**, the server uses its **default-artifact-root** (`mlflow-artifacts:/` by default, which resolves to `./mlartifacts/` on disk where you launched `mlflow ui`) and appends the new experiment's id: `mlflow-artifacts:/<experiment_id>`. That's why we got `mlartifacts/1/`.  
- **If you do pass it**, your value is used verbatim — no `experiment_id` is appended — and that's what ends up as the on-disk parent directory.

Two important constraints when applying this fix:

1. **`mlflow.set_experiment(name)` does NOT accept `artifact_location`.** It only creates the experiment with defaults if it's missing. To control the location you must call `mlflow.create_experiment(...)` explicitly *before* `set_experiment(...)`.  
2. **`artifact_location` is set once, at creation, and is effectively immutable through the public API.** Renaming an experiment is fine; *relocating* one is not. So the existing `"MLflow Quickstart"` experiment from the main tutorial above, which already has `artifact_location='mlflow-artifacts:/1'`, will keep that path forever.

The cleanest demonstration is therefore to **create a brand-new experiment** with a friendly `artifact_location` and run training under it. The code cells below do exactly that.


In [10]:
# Create a new experiment with an explicit, human-readable artifact_location.
# This leaves the original "MLflow Quickstart" experiment (and its mlartifacts/1/
# directory) untouched — we are demonstrating the parameter, not migrating data.
#
# Wrapped in try/except on RESOURCE_ALREADY_EXISTS so the cell is **idempotent**:
# the first run creates the experiment, every subsequent run silently no-ops
# instead of raising. Without this guard, re-running this cell raises
# `MlflowException: RESOURCE_ALREADY_EXISTS`, which would interrupt the tutorial
# flow — see principle 7 in the mlflow-tutorial-improve skill.

from mlflow.exceptions import RestException

friendly_experiment_name = "MLflow Quickstart (friendly path)"

try:
    mlflow.create_experiment(
        name=friendly_experiment_name,
        artifact_location="mlflow-artifacts:/mlflow-quickstart-friendly",
    )
except RestException as e:
    if "RESOURCE_ALREADY_EXISTS" not in str(e):
        raise  # some other error — surface it

# Make it the active experiment for subsequent `start_run` calls.
mlflow.set_experiment(friendly_experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/mlflow-quickstart-friendly', creation_time=1779435042123, experiment_id='2', last_update_time=1779435042123, lifecycle_stage='active', name='MLflow Quickstart (friendly path)', tags={}, trace_location=None, workspace='default'>

### Demonstration: log a model under the new experiment

Re-run the same iris training and `log_model` under the friendly experiment. The training code below is identical to the main tutorial's run cell except that there's no `registered_model_name=` — the goal here is only to show the on-disk layout change, not to add another version to the `tracking-quickstart` registered model.


In [11]:
with mlflow.start_run(run_name="iris-friendly-path-demo"):
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.set_tag("Training Info", "Friendly artifact_location demo")

    signature = infer_signature(X_train, lr.predict(X_train))

    friendly_model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        name="iris_model",
        signature=signature,
        input_example=X_train,
        serialization_format="skops",
    )

print("Model URI:", friendly_model_info.model_uri)


2026/05/22 10:40:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run iris-friendly-path-demo at: http://127.0.0.1:5001/#/experiments/2/runs/2c96c4a2969745e187f2cc5fc754540a
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
Model URI: models:/m-2873e49e92a445768c180a10330bd4b8


### What changed on disk

Inside `src/mlartifacts/`, you now have a **second top-level directory** next to the original `1/`. Where the default-named experiment landed at `mlartifacts/1/` (the integer `experiment_id`), this new one landed at `mlartifacts/mlflow-quickstart-friendly/` — the literal string we passed as `artifact_location`. Everything *under* the new directory (`models/m-<uuid>/artifacts/MLmodel`, `model.skops`, `conda.yaml`, …) is identical in shape to what cell `### Directory structure…` showed for the first run.

The MLflow UI now lists three experiments:

| `experiment_id` | name | `artifact_location` |
|---|---|---|
| `0` | `Default` | `mlflow-artifacts:/0` |
| `1` | `MLflow Quickstart with Logistic Regression` | `mlflow-artifacts:/1` |
| `2` | `MLflow Quickstart (friendly path)` | `mlflow-artifacts:/mlflow-quickstart-friendly` |

Note that the new experiment's **`experiment_id` is still `2`** — `artifact_location` only changes the *filesystem* path, not the integer id. Anywhere MLflow's APIs return an experiment id (run URLs, the registry's underlying tables), you'll still see numbers.


### Alternatives if you really want to keep the original name

Two heavier options, both with caveats:

1. **Soft-delete the existing experiment, then create a new one with the same name and a friendly `artifact_location`:**

   ```python
   exp = mlflow.get_experiment_by_name("MLflow Quickstart")
   if exp is not None:
       mlflow.delete_experiment(exp.experiment_id)  # soft delete
   mlflow.create_experiment(
       name="MLflow Quickstart",
       artifact_location="mlflow-artifacts:/mlflow-quickstart",
   )
   ```

   `delete_experiment` is a soft-delete — runs and metrics stay in the SQLite backend (`lifecycle_stage = 'deleted'`) and the artifact files stay on disk. Worse, MLflow won't let you create a new experiment with the same name while the trashed one still exists; you have to purge it first with `mlflow gc --backend-store-uri sqlite:///mlflow.db`, which permanently deletes both the DB rows and the `mlartifacts/<id>/` directory.

2. **Configure a server-wide default-artifact-root that already uses names**, e.g. start the server with `mlflow ui --default-artifact-root <path>` and ensure new experiments get created with custom `artifact_location` from day one. This still doesn't help experiments that already exist with the old default.

### References

- [`mlflow.create_experiment` — `artifact_location` parameter](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.create_experiment)  
- [Artifact Stores — default-artifact-root and `mlflow-artifacts:` proxy](https://mlflow.org/docs/latest/ml/tracking/artifact-stores/)  
- [MLflow 3 `LoggedModel` concept](https://mlflow.org/docs/latest/ml/model/)
